# Автоматическое дифферинцирование с помощью `torch.autograd`

При обучении нейронных сетей наиболее часто используемым алгоритмом является обратное распространение. В этом алгоритме параметры (модель веса) регулируется в соответствии с градиентом функции потери к данному параметру.

Чтобы вычислить эти градиенты, PyTorch имеет встроенный движок дифференциации, который называется `torch.autograd`. Он поддерживает автоматическое вычисление градиента для любого вычислительного графа.

Рассмотрим простейшую однослойную нейронную сеть с вводом `x`, параметры `w` и `b` и некоторые потери функции. Это может быть определено в PyTorch в следующем порядке:

In [2]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [3]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x113134bb0>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x112efe9e0>


## Вычисление градиента

In [4]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.0513, 0.0887, 0.0629],
        [0.0513, 0.0887, 0.0629],
        [0.0513, 0.0887, 0.0629],
        [0.0513, 0.0887, 0.0629],
        [0.0513, 0.0887, 0.0629]])
tensor([0.0513, 0.0887, 0.0629])


## Отключение отслеживания градиента

In [5]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


Или через `detach()`

In [6]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

False


Причины, по которым возможно отключение отслеживание градиента:

- Чтобы отметить некоторые параметры в нейронной сети как замороженные.

- Ускорить вычисления, когда есть только `forward`, потому что вычисления на тензорах, которые не отслеживают градиенты будут более эффективными.

